# Imports

In [6]:
import os
import pickle

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import PyMuPDFLoader

## Functions

In [2]:
def load_embeddings():
    
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    return embeddings

def create_vector_store(chunks, embeddings):

    db = FAISS.from_documents(chunks, embeddings)

    db.save_local("../vector_store")

    return db

def load_pdfs(folder_path):

    docs = []

    for file in os.listdir(folder_path):

        if file.endswith(".pdf"):

            path = os.path.join(folder_path, file)

            loader = PyMuPDFLoader(path)

            documents = loader.load()

            docs.extend(documents)

    return docs

def recursive_chunking(docs):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=100,
        separators=["\n\n", "\n", ".", " "]
    )

    return splitter.split_documents(docs)

def semantic_chunking(docs):

    embeddings = load_embeddings()
    
    splitter = SemanticChunker(embeddings)

    return splitter.split_documents(docs)

def create_vector_store(chunks):

    embeddings = load_embeddings()

    db = FAISS.from_documents(
        chunks,
        embeddings
    )

    db.save_local("../vector_store")

# Create Database

In [3]:
docs = load_pdfs("../data/raw")

In [4]:
chunks = semantic_chunking(docs)

/tmp/ipykernel_8037/3074833534.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
create_vector_store(chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
with open('../data/interim/semantic_chunks.pkl', 'wb') as file:
    pickle.dump(chunks, file)